# imports

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[0] ## Path().resolve() - notebook location and parents[0] goes from notebooks/ to project_root/
print(project_root)
sys.path.append(str(project_root))

In [ ]:
import glob
import pandas as pd
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from mpl_toolkits.basemap import Basemap

# functions

In [ ]:
def get_zoned_df(appended_data):
    '''
    returns multiple dataframes corresponding to 4 different basins
    '''
    
    zone_ARCTIC = appended_data.loc[appended_data['nav_lat'] > 70.0]
    zone_ARCTIC['zone'] = 'ARCTIC'
        
    zone_NORTH_ATLANTIC= appended_data.loc[(appended_data['nav_lon'] >= -75.0) & (appended_data['nav_lon'] <= 0.0)]
    zone_NORTH_ATLANTIC = zone_NORTH_ATLANTIC.loc[(zone_NORTH_ATLANTIC['nav_lat'] >= 10) & (zone_NORTH_ATLANTIC['nav_lat'] <= 70)]
    zone_NORTH_ATLANTIC['zone'] = 'NORTH_ATLANTIC'
    
    zone_EQ= appended_data.loc[(appended_data['nav_lat'] >= -10.0) & (appended_data['nav_lat'] <= 10.0)]
    zone_EQ_PACIFIC_1 = zone_EQ.loc[(zone_EQ['nav_lon'] >= 105.0) & (zone_EQ['nav_lon'] <= 180.0)]
    zone_EQ_PACIFIC_2 = zone_EQ.loc[(zone_EQ['nav_lon'] >= -180.0) & (zone_EQ['nav_lon'] <= -80.0)]
    zone_EQ_PACIFIC = pd.concat([zone_EQ_PACIFIC_1, zone_EQ_PACIFIC_2])
    zone_EQ_PACIFIC['zone'] = 'EQ_PACIFIC'
    
    zone_SOUTHERN_OCEAN = appended_data.loc[appended_data['nav_lat'] <= -45]
    zone_SOUTHERN_OCEAN['zone'] = 'SOUTHERN_OCEAN'
    
    return zone_ARCTIC, zone_NORTH_ATLANTIC, zone_EQ_PACIFIC, zone_SOUTHERN_OCEAN

In [ ]:
def assign_basins(nav_lat, nav_lon):
    '''
    Assigns basins to individual latitude and longitude values. 
    !!! Better use it as a lambda function.
    
    df['basin'] = df.apply(lambda row: assign_basins(row['nav_lat'], row['nav_lon']), axis=1)
    '''
    if nav_lat > 70.0:
        return 'ARCTIC'
    elif -75.0 <= nav_lon <= 0.0 and 10 <= nav_lat <= 70: 
        return 'NORTH_ATLANTIC'
    elif -10.0 <= nav_lat <= 10.0:
        if 105.0 <= nav_lon <= 180.0 or -180.0 <= nav_lon <= -80.0:
            return 'EQ_PACIFIC'
    elif nav_lat <= -45:
        return 'SOUTHERN_OCEAN'
    else:
        return 'OTHER'

# Load test files

In [ ]:
months_text = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

In [ ]:
sim_num=1
dl_model_name = 'TCN'
list_of_test_years = glob.glob(f"{dl_model_name}/tcn_attention_with_calib/*.parquet")
list_of_test_years

In [ ]:
len(list_of_test_years)

In [ ]:
test_years = range(2000,2019)

df_list = []

for test_yr in test_years:
    # print(test_yr)
    fn = [p for p in list_of_test_years
          if Path(p).stem.endswith(f"_{test_yr}")]
    print(fn[0])
    if len(fn) != 1:
        raise ValueError(f"Year {test_yr}: expected 1 file, got {len(fn)}: {fn[:3]}")
    data_df_yr = pd.read_parquet(fn[0])
    df_list.append(data_df_yr)

In [ ]:
df_all_test_years = pd.concat(df_list)
df_all_test_years.tail()

In [ ]:
df_all_test_years.columns

# Maps

In [ ]:
required_month = 7
required_year = 2018

_df_ = df_all_test_years.loc[(df_all_test_years['month'] == required_month) & (df_all_test_years['year'] == required_year)]
_df_['reconstruction_error_signed'] = _df_['co2flux_pre_reconstructed'] - _df_['co2flux_pre_simulated']
_df_['reconstruction_error_absolute'] = _df_['reconstruction_error_signed'].abs()

In [ ]:
%%time
fig, axes = plt.subplots(1, 3, figsize=(40, 18), dpi=200)
axes = axes.flatten()  # flatten 2D array of axes into a list

plot_settings = [
    (u"CO\u2082 flux simulated","co2flux_pre_simulated", "RdBu_r", -8, 8),
    (u"CO\u2082 flux reconstructed", "co2flux_pre_reconstructed", "RdBu_r", -8, 8),
    ("Point-wise reconstruction error","reconstruction_error_signed", "PiYG", -8, 8),
    # ("reconstruction_error_absolute", "Purples", 0, 3),
    
]

for ax, (title, plot_col, cmap, vmin, vmax) in zip(axes, plot_settings):

    world_map = Basemap(
        projection='cyl', resolution='c',
        llcrnrlat=-90, urcrnrlat=90,
        llcrnrlon=-180, urcrnrlon=180,
        ax=ax
    )

    sc = world_map.scatter(
        _df_['nav_lon'], _df_['nav_lat'],
        s=5, 
        c=_df_[plot_col],
        vmin=vmin, vmax=vmax,
        cmap=cmap, edgecolors='none'
    )

    world_map.fillcontinents(color='black')

    ax.set_title(f"{title} in {required_year} - {months_text[required_month-1]}", fontsize=35)

    cbar = fig.colorbar(sc, ax=ax, shrink=0.3, pad=0.02)
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label(f"(mol m⁻² yr⁻¹)", fontsize=25)
    # cbar.set_label(f"+ve into and -ve out of the ocean (mol m⁻² yr⁻¹)", fontsize=15)

# Reduce spacing between rows and columns
plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.savefig(f"map_{dl_model_name}_{required_year}_{required_month}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# required_month = 7
# _df_ = df_all_test_years.loc[df_all_test_years['month'] == required_month]
# _df_['se'] = (_df_['co2flux_pre_reconstructed'] - _df_['co2flux_pre_simulated'])**2
# _df_g = _df_.groupby(["nav_lat", "nav_lon",]).mean()

required_month = 7
_df_ = df_all_test_years.loc[df_all_test_years['month'] == required_month].copy()

_df_['se'] = (_df_['co2flux_pre_reconstructed'] - _df_['co2flux_pre_simulated'])**2 ## square the error

_df_g = (
    _df_
    .groupby(['nav_lat', 'nav_lon'])
    [['se', 'co2flux_pre_simulated', 'co2flux_pre_reconstructed']]
    .mean() ## take the mean
)

_df_g['rmse'] = np.sqrt(_df_g['se']) ## sq. root
# _df_g = _df_g.drop(columns='se')
_df_g.head()

In [ ]:
_df_g = _df_g.reset_index()
_df_g

In [ ]:
fig = plt.figure(figsize=(8, 5), edgecolor='w', dpi=250)
world_map = Basemap(projection='cyl', resolution='c',
            llcrnrlat=-90, urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180, )

feat = 'co2flux_pre_simulated'

world_map_scatter =world_map.scatter(_df_g['nav_lon'], _df_g['nav_lat'],
                                     s = 1, 
                                     c = _df_g[feat],
                                     vmin=-8, vmax =8, 
                                     cmap='RdBu_r', 
                                     edgecolors='none',
                                    # color = 'red',
                                    )

cbar = plt.colorbar(world_map_scatter, shrink = 0.7, pad=0.01)
cbar.ax.tick_params(labelsize=8) ## Hide axis labels
cbar.set_label('(mol m⁻² yr⁻¹)', fontsize=16)


# m.shadedrelief()
## Fill the land mass and lakes
world_map.fillcontinents(color='black') #color_lake='aqua'
if required_month == 1:
    m_name = 'January'
elif required_month == 7:
    m_name = 'July'
plt.title(f"Mean CO₂ flux simulated in {m_name}", fontsize=18)
plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 5), edgecolor='w', dpi=250)
world_map = Basemap(projection='cyl', resolution='c',
            llcrnrlat=-90, urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180, )

feat = 'co2flux_pre_reconstructed'

world_map_scatter =world_map.scatter(_df_g['nav_lon'], _df_g['nav_lat'],
                                     s = 1, 
                                     c = _df_g[feat],
                                     vmin=-8, vmax =8, 
                                     cmap='RdBu_r', 
                                     edgecolors='none',
                                    # color = 'red',
                                    )

cbar = plt.colorbar(world_map_scatter, shrink = 0.7, pad=0.01)
cbar.ax.tick_params(labelsize=8) ## Hide axis labels
cbar.set_label('(mol m⁻² yr⁻¹)', fontsize=16)


# m.shadedrelief()
## Fill the land mass and lakes
world_map.fillcontinents(color='black') #color_lake='aqua'
if required_month == 1:
    m_name = 'January'
    fs=16
elif required_month == 7:
    m_name = 'July'
    fs=16
plt.title(f"Mean CO₂ flux reconstructed in {m_name} by TCN - Attention", fontsize=fs)
plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 5), edgecolor='w', dpi=250)
world_map = Basemap(projection='cyl', resolution='c',
            llcrnrlat=-90, urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180, )

feat = 'rmse'

world_map_scatter =world_map.scatter(_df_g['nav_lon'], _df_g['nav_lat'],
                                     s = 1, 
                                     c = _df_g[feat],
                                     vmin=0, vmax =8, 
                                     cmap='Greens', 
                                     edgecolors='none',
                                    # color = 'red',
                                    )

cbar = plt.colorbar(world_map_scatter, shrink = 0.7, pad=0.01)
cbar.ax.tick_params(labelsize=8) ## Hide axis labels
cbar.set_label('(mol m⁻² yr⁻¹)', fontsize=16)


# m.shadedrelief()
## Fill the land mass and lakes
world_map.fillcontinents(color='black') #color_lake='aqua'
if required_month == 1:
    m_name = 'January'
elif required_month == 7:
    m_name = 'July'
plt.title(f"Point-wise RMSE in {m_name} by TCN - Attention", fontsize=18)
plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.show()

In [ ]:
%%time
fig, axes = plt.subplots(1, 3, figsize=(60, 27), dpi=200)
axes = axes.flatten()  # flatten 2D array of axes into a list

plot_settings = [
    (u"Mean CO\u2082 flux simulated","co2flux_pre_simulated", "RdBu_r", -8, 8),
    (u"Mean CO\u2082 flux reconstructed by TCN-Atention", "co2flux_pre_reconstructed", "RdBu_r", -8, 8),
    ("Point-wise RMSE","rmse", "Greens", 0, 8),
    # ("reconstruction_error_absolute", "Purples", 0, 3),
    
]

for ax, (title, plot_col, cmap, vmin, vmax) in zip(axes, plot_settings):

    world_map = Basemap(
        projection='cyl', resolution='c',
        llcrnrlat=-90, urcrnrlat=90,
        llcrnrlon=-180, urcrnrlon=180,
        ax=ax
    )

    sc = world_map.scatter(
        _df_g['nav_lon'], _df_g['nav_lat'],
        s=5, 
        c=_df_g[plot_col],
        vmin=vmin, vmax=vmax,
        cmap=cmap, edgecolors='none'
    )

    world_map.fillcontinents(color='black')

    ax.set_title(f"{title} in {months_text[required_month-1]}", fontsize=38)

    cbar = fig.colorbar(sc, ax=ax, shrink=0.3, pad=0.02)
    cbar.ax.tick_params(labelsize=35)
    cbar.set_label(f"(mol m⁻² yr⁻¹)", fontsize=40)
    # cbar.set_label(f"+ve into and -ve out of the ocean (mol m⁻² yr⁻¹)", fontsize=15)

# Reduce spacing between rows and columns
plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.savefig(f"map_{dl_model_name}_month_{required_month}_19testyears.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
%%time
fig, axes = plt.subplots(2, 3, figsize=(60, 54), dpi=200)
axes = axes.reshape(2, 3)

dfs = [_df_g, _df_g_2]
row_titles = ["Dataset 1", "Dataset 2"]  # change to model names if needed

plot_settings = [
    (u"Mean CO\u2082 flux simulated","co2flux_pre_simulated", "RdBu_r", -8, 8),
    (u"Mean CO\u2082 flux reconstructed by TCN-Attention", "co2flux_pre_reconstructed", "RdBu_r", -8, 8),
    ("Point-wise RMSE","rmse", "Greens", 0, 8),
]

for row, (df_plot, row_name) in enumerate(zip(dfs, row_titles)):

    for col, (title, plot_col, cmap, vmin, vmax) in enumerate(plot_settings):

        ax = axes[row, col]

        world_map = Basemap(
            projection='cyl',
            resolution='c',
            llcrnrlat=-90, urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180,
            ax=ax
        )

        sc = world_map.scatter(
            df_plot['nav_lon'],
            df_plot['nav_lat'],
            s=5,
            c=df_plot[plot_col],
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
            edgecolors='none'
        )

        world_map.fillcontinents(color='black')

        ax.set_title(
            f"{title} ({row_name}) in {months_text[required_month-1]}",
            fontsize=38
        )

        cbar = fig.colorbar(sc, ax=ax, shrink=0.3, pad=0.02)
        cbar.ax.tick_params(labelsize=35)
        cbar.set_label("(mol m⁻² yr⁻¹)", fontsize=40)

plt.tight_layout(h_pad=-25, w_pad=0.2)

plt.savefig(
    f"map_{dl_model_name}_month_{required_month}_19testyears.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Error flows

In [ ]:
%%time
sim = 'co2flux_pre_simulated'
rec = 'co2flux_pre_reconstructed'

# RMSE per (year, month)
rmse_ym = (
    df_all_test_years
    .assign(se=(df_all_test_years[rec] - df_all_test_years[sim])**2)
    .groupby(['year', 'month'])['se']
    .mean()
    .pipe(np.sqrt)
    .reset_index(name='rmse')
)

# year x month table 
rmse_mat = (
    rmse_ym
    .pivot(index='year', columns='month', values='rmse')
    .reindex(columns=range(1, 13))   
    .sort_index()                   
)

In [ ]:
rmse_mat

In [ ]:
# rename columns and index
rmse_mat.columns = months_text
rmse_mat.index = rmse_mat.index.astype(int).astype(str)

fig, ax = plt.subplots(figsize=(10,6), dpi=200)

sns.heatmap(
    rmse_mat,
    cmap="YlGn",
    annot=True,
    fmt=".2f",
    vmin=0.4,
    vmax=0.7,
    annot_kws={"size":13},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "RMSE (mol m⁻² yr⁻¹)"},
    ax=ax
)

ax.set_xlabel("Month", fontsize=20)
ax.set_ylabel("Year", fontsize=20)
ax.set_title(f"{dl_model_name}-Attention monthly test RMSE in global ocean ", fontsize=20)

ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=14)

# increase colorbar font sizes
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=17)
cbar.set_label("RMSE (mol m⁻² yr⁻¹)", fontsize=20)

plt.tight_layout()


plt.savefig(f"{dl_model_name}_monthly_rmse_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## Basin

In [ ]:
_, df_natl, _, df_so = get_zoned_df(appended_data = df_all_test_years)

In [ ]:
df_so.head()

In [ ]:
%%time
sim = 'co2flux_pre_simulated'
rec = 'co2flux_pre_reconstructed'

# RMSE per (year, month)
rmse_ym = (
    df_natl
    .assign(se=(df_natl[rec] - df_natl[sim])**2)
    .groupby(['year', 'month'])['se']
    .mean()
    .pipe(np.sqrt)
    .reset_index(name='rmse')
)

# year x month table 
rmse_mat = (
    rmse_ym
    .pivot(index='year', columns='month', values='rmse')
    .reindex(columns=range(1, 13))   
    .sort_index()                   
)

# rename columns and index
rmse_mat.columns = months_text
rmse_mat.index = rmse_mat.index.astype(int).astype(str)

fig, ax = plt.subplots(figsize=(10,6), dpi=200)

sns.heatmap(
    rmse_mat,
    cmap="YlGn",
    annot=True,
    fmt=".2f",
    vmin=0.4,
    vmax=0.7,
    annot_kws={"size":13},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "RMSE (mol m⁻² yr⁻¹)"},
    ax=ax
)

ax.set_xlabel("Month", fontsize=20)
ax.set_ylabel("Year", fontsize=20)
ax.set_title(f"{dl_model_name}-Attention monthly test RMSE in NATL basin", fontsize=20)

ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=14)

# increase colorbar font sizes
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=17)
cbar.set_label("RMSE (mol m⁻² yr⁻¹)", fontsize=20)

plt.tight_layout()


plt.savefig(f"{dl_model_name}_monthly_rmse_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
%%time
sim = 'co2flux_pre_simulated'
rec = 'co2flux_pre_reconstructed'

# RMSE per (year, month)
rmse_ym = (
    df_so
    .assign(se=(df_so[rec] - df_so[sim])**2)
    .groupby(['year', 'month'])['se']
    .mean()
    .pipe(np.sqrt)
    .reset_index(name='rmse')
)

# year x month table 
rmse_mat = (
    rmse_ym
    .pivot(index='year', columns='month', values='rmse')
    .reindex(columns=range(1, 13))   
    .sort_index()                   
)

# rename columns and index
rmse_mat.columns = months_text
rmse_mat.index = rmse_mat.index.astype(int).astype(str)

fig, ax = plt.subplots(figsize=(10,6), dpi=200)

sns.heatmap(
    rmse_mat,
    cmap="YlGn",
    annot=True,
    fmt=".2f",
    vmin=0.4,
    vmax=0.7,
    annot_kws={"size":13},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "RMSE (mol m⁻² yr⁻¹)"},
    ax=ax
)

ax.set_xlabel("Month", fontsize=20)
ax.set_ylabel("Year", fontsize=20)
ax.set_title(f"{dl_model_name}-Attention monthly test RMSE in SO basin", fontsize=20)

ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=14)

# increase colorbar font sizes
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)
cbar.set_label("RMSE (mol m⁻² yr⁻¹)", fontsize=20)

plt.tight_layout()

plt.savefig(f"{dl_model_name}monthly_rmse_heatmap_SO.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("STOP")

## Line plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), dpi=200)

years = rmse_mat.index.to_numpy()
months = rmse_mat.columns.to_numpy()

cmap = plt.cm.get_cmap('tab20', len(years))   # 20 distinct colors - matplotlib.colormaps[name]

for i, (year, row) in enumerate(rmse_mat.iterrows()):
#     ax.scatter(months, row.to_numpy(), # plot, scatter
#             color=cmap(i),
#             label=int(year),
#             linewidth=1.5)

    yvals = row.to_numpy()
    color = cmap(i)

    # thin line
    ax.plot(months, yvals, color=color, linewidth=0.8, alpha=0.8)

    # bubbles
    ax.scatter(months, yvals, color=color, s=18, label=int(year), zorder=3, alpha=1.0)

# ax.tick_params(size=20)
ax.set_xlabel('Month', fontsize=20)
ax.set_ylabel('RMSE (mol m⁻² yr⁻¹)', fontsize=20)
# ax.set_xticks(months_text)
ax.set_title('Monthly Test RMSE from TCN-Attention model', fontsize=20)

ax.legend(title='Year', ncol=6, fontsize=8, frameon=True, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("{dl_model_name}_monthly_test_rmse_over_test_years.png", dpi=300, bbox_inches="tight")
plt.show()

# Means

In [ ]:
%%time

global_monthly_mean_sim = []
global_monthly_std_sim = []

global_monthly_mean_rec = []
global_monthly_std_rec = []

for yr in range(2000,2019):
    for m in range(1,13):
        _df_m = df_all_test_years.loc[(df_all_test_years['year']==yr) & (df_all_test_years['month']==m)]
        global_monthly_mean_sim.append(_df_m['co2flux_pre_simulated'].mean())
        global_monthly_std_sim.append(_df_m['co2flux_pre_simulated'].std())
        global_monthly_mean_rec.append(_df_m['co2flux_pre_reconstructed'].mean())
        global_monthly_std_rec.append(_df_m['co2flux_pre_reconstructed'].std())

In [ ]:
len(global_monthly_mean_sim)

In [ ]:
global_annual_mean_sim = []
global_annual_std_sim = []

global_annual_mean_rec = []
global_annual_std_rec = []

for yr in range(2000,2019):
    _df_ = df_all_test_years.loc[df_all_test_years['year']==yr]
    global_annual_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    global_annual_std_sim.append(_df_['co2flux_pre_simulated'].std())
    global_annual_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    global_annual_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

In [ ]:
years = np.arange(2000, 2019)

mean_sim = np.array(global_annual_mean_sim)
std_sim  = np.array(global_annual_std_sim)

mean_rec = np.array(global_annual_mean_rec)
std_rec  = np.array(global_annual_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

# Simulated
plt.plot(years, mean_sim, label='Simulated', lw=1, color='cornflowerblue')
# plt.fill_between(years,
#                  mean_sim - std_sim,
#                  mean_sim + std_sim,
#                  alpha=0.15)

# Reconstructed
plt.plot(years, mean_rec, label='Reconstructed', lw=1, color='tomato')
# plt.fill_between(years,
#                  mean_pred - std_pred,
#                  mean_pred + std_pred,
#                  alpha=0.15)

plt.xlabel("Year")
plt.ylabel("CO₂ flux pre. (mol m⁻² yr⁻¹)")
plt.title("Global annual mean (2000-2018)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
global_seasonal_mean_sim = []
global_seasonal_std_sim = []

global_seasonal_mean_rec = []
global_seasonal_std_rec = []

for m in range(1,13):
    _df_ = df_all_test_years.loc[df_all_test_years['month']==m]
    global_seasonal_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    global_seasonal_std_sim.append(_df_['co2flux_pre_simulated'].std())
    global_seasonal_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    global_seasonal_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

In [ ]:
months = np.arange(1, 13)

seasonal_mean_sim = np.array(global_seasonal_mean_sim)
seasonal_std_sim  = np.array(global_seasonal_std_sim)
seasonal_mean_rec = np.array(global_seasonal_mean_rec)
seasonal_std_rec  = np.array(global_seasonal_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

months_text = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

# Simulated
plt.plot(months_text, seasonal_mean_sim, label='Simulated', lw=1, color='cornflowerblue')
plt.fill_between(months_text,
                 seasonal_mean_sim - seasonal_std_sim,
                 seasonal_mean_sim + seasonal_std_sim,
                 alpha=0.15)

# Reconstructed
plt.plot(months_text, seasonal_mean_rec, label='Reconstructed', lw=1, color='tomato')
plt.fill_between(months_text,
                 seasonal_mean_rec - seasonal_std_rec,
                 seasonal_mean_rec + seasonal_std_rec,
                 alpha=0.15)

plt.xlabel("Months")
plt.ylabel("CO₂ flux pre. (mol m⁻² yr⁻¹)")
plt.title("Global seasonal mean (2000-2018)")
plt.legend()
plt.tight_layout()
plt.show()

# Reconstruction in Basin analysis

In [ ]:
df_all_test_years

In [ ]:
_, df_natl, _, df_so = get_zoned_df(appended_data = df_all_test_years)

## NATL

In [ ]:
df_natl

In [ ]:
# Derived uncertainty (optional but usually what you want)
logvar = df_natl['co2flux_pre_reconstructed_log_var'].values
var   = np.exp(logvar)             # (Ns, 1)
sigma = np.sqrt(var)  

In [ ]:
natl_monthly_mean_sim = []
natl_monthly_std_sim = []

natl_monthly_mean_rec = []
natl_monthly_std_rec = []

plot_lables = []
count = 0
for yr in range(2000,2019):
    for m in range(1,13):
        _df_ = df_natl.loc[(df_natl['year']==yr) & (df_natl['month']==m)]
        natl_monthly_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
        natl_monthly_std_sim.append(_df_['co2flux_pre_simulated'].std())

        logvar = _df_['co2flux_pre_reconstructed_log_var'].values
        var   = np.exp(logvar)             # (Ns, 1)
        sigma = np.sqrt(var)  
        natl_monthly_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
        natl_monthly_std_rec.append(np.mean(sigma))
        # natl_monthly_std_rec.append(_df_['co2flux_pre_reconstructed'].std())
        count = count + 1
        plot_lables.append(count)
        # plot_lables.append(f"{yr}-{m}")

In [ ]:
mean_monthly_sim = np.array(natl_monthly_mean_sim)
std_monthly_sim  = np.array(natl_monthly_std_sim)
mean_monthly_rec = np.array(natl_monthly_mean_rec)
std_monthly_rec  = np.array(natl_monthly_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

# Simulated
plt.plot(plot_lables, mean_monthly_sim, label='simulated', lw=1, color='dodgerblue')
# plt.fill_between(years,
#                  mean_sim - std_sim,
#                  mean_sim + std_sim,
#                  alpha=0.15)

# Reconstructed
plt.plot(plot_lables, mean_monthly_rec, label='reconstructed with prediction boundary', lw=1, color='brown')
plt.fill_between(plot_lables,
                 mean_monthly_rec - std_monthly_rec,
                 mean_monthly_rec + std_monthly_rec,
                 alpha=0.15, color='brown')

plt.xlabel("Year", fontsize=20)
plt.ylabel("CO₂ flux (mol m⁻² yr⁻¹)", fontsize=20)
plt.title("NATL monthly mean CO₂ flux (2000-2018)", fontsize=20)

# tick positions at start of each year
years = list(range(2000, 2019))
year_ticks = [i*12 + 1 for i in range(len(years))]
plt.xticks(year_ticks, years, fontsize=12, rotation=30, ha='right')
plt.yticks(fontsize=16)

plt.legend(fontsize=11, loc='best', frameon=False)
# plt.legend(fontsize=12, loc='best', bbox_to_anchor=(1, 0.5), frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
natl_annual_mean_sim = []
natl_annual_std_sim = []

natl_annual_mean_rec = []
natl_annual_std_rec = []

for yr in range(1990,2019):
    _df_ = df_natl.loc[df_natl['year']==yr]
    natl_annual_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    natl_annual_std_sim.append(_df_['co2flux_pre_simulated'].std())
    natl_annual_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    natl_annual_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

In [ ]:
years = np.arange(1990, 2019)

mean_sim = np.array(natl_annual_mean_sim)
std_sim  = np.array(natl_annual_std_sim)

mean_rec = np.array(natl_annual_mean_rec)
std_rec  = np.array(natl_annual_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

# Simulated
plt.plot(years, mean_sim, label='Simulated', lw=1, color='cornflowerblue')
# plt.fill_between(years,
#                  mean_sim - std_sim,
#                  mean_sim + std_sim,
#                  alpha=0.15)

# Reconstructed
plt.plot(years, mean_rec, label='Reconstructed', lw=1, color='tomato')
# plt.fill_between(years,
#                  mean_pred - std_pred,
#                  mean_pred + std_pred,
#                  alpha=0.15)

plt.xlabel("Year")
plt.ylabel("CO₂ flux pre. (mol m⁻² yr⁻¹)")
plt.title(f"{basin_of_interest} annual mean CO₂ flux (1990-2018)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
natl_seasonal_mean_sim = []
natl_seasonal_std_sim = []

natl_seasonal_mean_rec = []
natl_seasonal_std_rec = []

for m in range(1,13):
    _df_ = df_natl.loc[df_natl['month']==m]
    natl_seasonal_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    natl_seasonal_std_sim.append(_df_['co2flux_pre_simulated'].std())
    natl_seasonal_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    natl_seasonal_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

In [ ]:
months = np.arange(1, 13)

natl_seasonal_mean_sim = np.array(natl_seasonal_mean_sim)
natl_seasonal_std_sim  = np.array(natl_seasonal_std_sim)
natl_seasonal_mean_rec = np.array(natl_seasonal_mean_rec)
natl_seasonal_std_rec  = np.array(natl_seasonal_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

months_text = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']


# Simulated
plt.plot(months_text, natl_seasonal_mean_sim, label='Simulated', lw=1, color='dodgerblue')
plt.fill_between(months_text,
                 natl_seasonal_mean_sim - natl_seasonal_std_sim,
                 natl_seasonal_mean_sim + natl_seasonal_std_sim,
                 alpha=0.15)

# Predicted
plt.plot(months_text, natl_seasonal_mean_rec, label='Reconstructed', lw=1, color='brown')
plt.fill_between(months_text,
                 natl_seasonal_mean_rec - natl_seasonal_std_rec,
                 natl_seasonal_mean_rec + natl_seasonal_std_rec,
                 alpha=0.15)

plt.xlabel("Months", fontsize=20)
plt.ylabel("CO₂ flux (mol m⁻² yr⁻¹)", fontsize=20)
plt.title(f"{basin_of_interest} Seasonal mean CO₂ flux (2000-2018)", fontsize=20)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)


plt.legend(fontsize=11, loc='best', frameon=False)

plt.tight_layout()
plt.show()

## SO

In [ ]:
df_so

In [ ]:
so_monthly_mean_sim = []
so_monthly_std_sim = []

so_monthly_mean_rec = []
so_monthly_std_rec = []

plot_lables = []
count = 0
for yr in range(2000,2019):
    for m in range(1,13):
        _df_ = df_so.loc[(df_so['year']==yr) & (df_so['month']==m)]
        so_monthly_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
        so_monthly_std_sim.append(_df_['co2flux_pre_simulated'].std())

        logvar = _df_['co2flux_pre_reconstructed_log_var'].values
        var   = np.exp(logvar)             # (Ns, 1)
        sigma = np.sqrt(var)  
        so_monthly_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
        so_monthly_std_rec.append(np.mean(sigma))

        count = count + 1
        plot_lables.append(count)
        # plot_lables.append(f"{yr}-{m}")

mean_monthly_sim = np.array(so_monthly_mean_sim)
std_monthly_sim  = np.array(so_monthly_std_sim)
mean_monthly_rec = np.array(so_monthly_mean_rec)
std_monthly_rec  = np.array(so_monthly_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

# Simulated
plt.plot(plot_lables, mean_monthly_sim, label='simulated', lw=1, color='dodgerblue')
# plt.fill_between(years,
#                  mean_sim - std_sim,
#                  mean_sim + std_sim,
#                  alpha=0.15)

# Reconstructed
plt.plot(plot_lables, mean_monthly_rec, label='reconstructed with prediction boundary', lw=1, color='brown')
plt.fill_between(plot_lables,
                 mean_monthly_rec - std_monthly_rec,
                 mean_monthly_rec + std_monthly_rec,
                 alpha=0.15, color='brown')

plt.xlabel("Year", fontsize=20)
plt.ylabel("CO₂ flux (mol m⁻² yr⁻¹)", fontsize=20)
plt.title("SO monthly mean CO₂ flux (2000-2018)", fontsize=20)

# tick positions at start of each year
years = list(range(2000, 2019))
year_ticks = [i*12 + 1 for i in range(len(years))]
plt.xticks(year_ticks, years, fontsize=12, rotation=30, ha='right')
plt.yticks(fontsize=16)

plt.legend(fontsize=11, loc='best', frameon=False)
# plt.legend(fontsize=12, loc='best', bbox_to_anchor=(1, 0.5), frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
so_seasonal_mean_sim = []
so_seasonal_std_sim = []

so_seasonal_mean_rec = []
so_seasonal_std_rec = []

for m in range(1,13):
    _df_ = df_so.loc[df_so['month']==m]
    so_seasonal_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    so_seasonal_std_sim.append(_df_['co2flux_pre_simulated'].std())
    so_seasonal_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    so_seasonal_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

months = np.arange(1, 13)

so_seasonal_mean_sim = np.array(so_seasonal_mean_sim)
so_seasonal_std_sim  = np.array(so_seasonal_std_sim)
so_seasonal_mean_rec = np.array(so_seasonal_mean_rec)
so_seasonal_std_rec  = np.array(so_seasonal_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

months_text = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']


# Simulated
plt.plot(months_text, so_seasonal_mean_sim, label='Simulated', lw=1, color='dodgerblue')
plt.fill_between(months_text,
                 so_seasonal_mean_sim - so_seasonal_std_sim,
                 so_seasonal_mean_sim + so_seasonal_std_sim,
                 alpha=0.15)

# Predicted
plt.plot(months_text, so_seasonal_mean_rec, label='Reconstructed', lw=1, color='brown')
plt.fill_between(months_text,
                 so_seasonal_mean_rec - so_seasonal_std_rec,
                 so_seasonal_mean_rec + so_seasonal_std_rec,
                 alpha=0.15)

plt.xlabel("Months", fontsize=20)
plt.ylabel("CO₂ flux (mol m⁻² yr⁻¹)", fontsize=20)
plt.title(f"SO Seasonal mean CO₂ flux (2000-2018)", fontsize=20)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)


plt.legend(fontsize=11, loc='best', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
so_annual_mean_sim = []
so_annual_std_sim = []
so_annual_mean_rec = []
so_annual_std_rec = []

for yr in range(1990,2019):
    _df_ = df_so.loc[df_so['year']==yr]
    so_annual_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    so_annual_std_sim.append(_df_['co2flux_pre_simulated'].std())
    so_annual_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    so_annual_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

In [ ]:
years = np.arange(1990, 2019)

mean_sim = np.array(so_annual_mean_sim)
std_sim  = np.array(so_annual_std_sim)

mean_rec = np.array(so_annual_mean_rec)
std_rec  = np.array(so_annual_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

# Simulated
plt.plot(years, mean_sim, label='Simulated', lw=1, color='cornflowerblue')
# plt.fill_between(years,
#                  mean_sim - std_sim,
#                  mean_sim + std_sim,
#                  alpha=0.25)

# Predicted
plt.plot(years, mean_rec, label='Reconstructed', lw=1, color='tomato')
# plt.fill_between(years,
#                  mean_rec - std_rec,
#                  mean_rec + std_rec,
#                  alpha=0.25)

plt.xlabel("Year")
plt.ylabel("CO₂ flux pre. (mol m⁻² yr⁻¹)")
plt.title("SO Annual mean CO₂ flux (1990-2018)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
so_seasonal_mean_sim = []
so_seasonal_std_sim = []
so_seasonal_mean_rec = []
so_seasonal_std_rec = []

for m in range(1,13):
    _df_ = df_so.loc[df_so['month']==m]
    so_seasonal_mean_sim.append(_df_['co2flux_pre_simulated'].mean())
    so_seasonal_std_sim.append(_df_['co2flux_pre_simulated'].std())
    so_seasonal_mean_rec.append(_df_['co2flux_pre_reconstructed'].mean())
    so_seasonal_std_rec.append(_df_['co2flux_pre_reconstructed'].std())

In [ ]:
months = np.arange(1, 13)

so_seasonal_mean_sim = np.array(so_seasonal_mean_sim)
so_seasonal_std_sim  = np.array(so_seasonal_std_sim)
so_seasonal_mean_rec = np.array(so_seasonal_mean_rec)
so_seasonal_std_rec  = np.array(so_seasonal_std_rec)

In [ ]:
plt.figure(figsize=(8,4), dpi=200)

# Simulated
plt.plot(months_text, so_seasonal_mean_sim, label='Simulated', lw=1, color='cornflowerblue')
plt.fill_between(months_text,
                 so_seasonal_mean_sim - so_seasonal_std_sim,
                 so_seasonal_mean_sim + so_seasonal_std_sim,
                 alpha=0.15)

# Predicted
plt.plot(months_text, so_seasonal_mean_rec, label='Reconstructed', lw=1, color='tomato')
plt.fill_between(months_text,
                 so_seasonal_mean_rec - so_seasonal_std_rec,
                 so_seasonal_mean_rec + so_seasonal_std_rec,
                 alpha=0.15)

plt.xlabel("Months")
plt.ylabel("CO₂ flux pre. (mol m⁻² yr⁻¹)")
plt.title("SO Seasonal mean CO₂ flux (1990-2018)")
plt.legend()
plt.tight_layout()
plt.show()

# Atention scores

In [ ]:
_, df_natl, _, df_so = get_zoned_df(appended_data = df_all_test_years)

In [ ]:
df_natl.head()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

lag_cols = ['attn_lag_5','attn_lag_4','attn_lag_3',
            'attn_lag_2','attn_lag_1','attn_lag_0']

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

labels = ['lag-5','lag-4','lag-3','lag-2','lag-1','lag-0']

df = df_natl

# Aggregate attention by month
heatmap_df = (
    df.groupby('month')[lag_cols]
      .mean()
      .reindex(range(1,13))
)

heatmap_df.index = month_labels

# --- single hue colormap: white -> dark blue ---
cmap = LinearSegmentedColormap.from_list(
    "white_blue",
    ["#ffffff", "#5e7ba6"]
)

plt.figure(figsize=(10,6), dpi=200)

ax = sns.heatmap(
    heatmap_df,
    cmap=cmap,
    vmin=0.0,
    vmax=0.01,
    annot=True,
    fmt=".4f",
    # cbar_kws={"label": "Mean attention"},
    annot_kws={"size":13},
    linewidths=0.5,
    linecolor="gray",
)

ax.set_xticklabels(labels)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=12)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=17)
cbar.set_label("Mean attention", fontsize=20)

plt.xlabel("Lag", fontsize=20)
plt.ylabel("Month", fontsize=20)
plt.title("Seasonal Mean Attention per lag in NATL", fontsize=18)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

lag_cols = ['attn_lag_5','attn_lag_4','attn_lag_3',
            'attn_lag_2','attn_lag_1','attn_lag_0']

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

labels = ['lag-5','lag-4','lag-3','lag-2','lag-1','lag-0']

df = df_so

# Aggregate attention by month
heatmap_df = (
    df.groupby('month')[lag_cols]
      .mean()
      .reindex(range(1,13))
)

heatmap_df.index = month_labels

# --- single hue colormap: white -> dark blue ---
cmap = LinearSegmentedColormap.from_list(
    "white_blue",
    ["#ffffff", "#5e7ba6"]
)

plt.figure(figsize=(10,6), dpi=200)

ax = sns.heatmap(
    heatmap_df,
    cmap=cmap,
    vmin=0.0,
    vmax=0.01,
    annot=True,
    fmt=".4f",
    # cbar_kws={"label": "Mean attention"},
    annot_kws={"size":13},
    linewidths=0.5,
    linecolor="gray",
)

ax.set_xticklabels(labels)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=12)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=17)
cbar.set_label("Mean attention", fontsize=20)

plt.xlabel("Lag", fontsize=20)
plt.ylabel("Month", fontsize=20)
plt.title("Seasonal Mean Attention per lag in SO", fontsize=18)

plt.tight_layout()
plt.show()

In [ ]:
lag_cols = ['attn_lag_5','attn_lag_4','attn_lag_3','attn_lag_2','attn_lag_1','attn_lag_0']

jan_mean = df_natl[df_natl['month'] == 1][lag_cols].mean()
jul_mean = df_natl[df_natl['month'] == 7][lag_cols].mean()

labels = ['lag-5','lag-4','lag-3','lag-2','lag-1','lag-0']

fig, ax = plt.subplots(1, 2, figsize=(10,5), dpi=200)

ax[0].pie(jan_mean, labels=labels, autopct='%1.1f%%',)
ax[0].set_title("January Lag Attention in NATL", fontsize=20)

ax[1].pie(jul_mean, labels=labels, autopct='%1.1f%%',)
ax[1].set_title("July Lag Attention in NATL", fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
lag_cols = ['attn_lag_5','attn_lag_4','attn_lag_3','attn_lag_2','attn_lag_1','attn_lag_0']

jan_mean = df_so[df_so['month'] == 1][lag_cols].mean()
jul_mean = df_so[df_so['month'] == 7][lag_cols].mean()

labels = ['lag-5','lag-4','lag-3','lag-2','lag-1','lag-0']

fig, ax = plt.subplots(1, 2, figsize=(10,5), dpi=200)

ax[0].pie(jan_mean, labels=labels, autopct='%1.1f%%')
ax[0].set_title("January Lag Attention in SO", fontsize=20)

ax[1].pie(jul_mean, labels=labels, autopct='%1.1f%%')
ax[1].set_title("July Lag Attention in SO", fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
lag_cols = [f'attn_lag_{i}' for i in range(6)]

mean_lag = df_natl[lag_cols].mean()
std_lag  = df_natl[lag_cols].std()

plt.errorbar(range(6), mean_lag[::-1], yerr=std_lag[::-1])

plt.xticks(range(6), ['5','4','3','2','1','0'])
plt.xlabel("Lag (months)")
plt.ylabel("Mean Attention")

In [ ]:
lag_cols = [f'attn_lag_{i}' for i in range(6)]

mean_lag = df_so[lag_cols].mean()
std_lag  = df_so[lag_cols].std()

plt.errorbar(range(6), mean_lag[::-1], yerr=std_lag[::-1])

plt.xticks(range(6), ['5','4','3','2','1','0'])
plt.xlabel("Lag (months)")
plt.ylabel("Mean Attention")

# Plot maps

In [ ]:
required_month = 7
required_year = 2018

_df_ = df_all_test_years.loc[(df_all_test_years['month'] == required_month) & (df_all_test_years['year'] == required_year)]
_df_['reconstruction_error_signed'] = _df_['co2flux_pre_reconstructed'] - _df_['co2flux_pre_simulated']
_df_['reconstruction_error_absolute'] = _df_['reconstruction_error_signed'].abs()

In [ ]:
_df_['reconstruction_error_signed'] = _df_['co2flux_pre_reconstructed'] - _df_['co2flux_pre_simulated']
_df_['reconstruction_error_absolute'] = _df_['reconstruction_error_signed'].abs()
# _df_['reconstruction_error_percentage'] = 100*(_df_['reconstruction_error_signed'] / _df_['co2flux_pre_simulated'])
_df_

In [ ]:
%%time
fig, axes = plt.subplots(2, 2, figsize=(30, 20), dpi=120)
axes = axes.flatten()  # flatten 2D array of axes into a list

plot_settings = [
    ("co2flux_pre_reconstructed", "RdYlBu_r", -8, 8),
    ("co2flux_pre_simulated", "RdYlBu_r", -8, 8),
    ("reconstruction_error_signed", "PiYG", -3, 3),
    ("reconstruction_error_absolute", "Purples", 0, 3),
    
]

for ax, (plot_col, cmap, vmin, vmax) in zip(axes, plot_settings):
    world_map = Basemap(
        projection='cyl', resolution='c',
        llcrnrlat=-90, urcrnrlat=90,
        llcrnrlon=-180, urcrnrlon=180,
        ax=ax
    )

    sc = world_map.scatter(
        _df_['nav_lon'], _df_['nav_lat'],
        s=5, 
        c=_df_[plot_col],
        vmin=vmin, vmax=vmax,
        cmap=cmap, edgecolors='none'
    )

    world_map.fillcontinents(color='black')

    ax.set_title(f"{plot_col} in {required_year}-{required_month}", fontsize=20)

    cbar = fig.colorbar(sc, ax=ax, shrink=0.3, pad=0.02)
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label(f"{plot_col} in month {required_month}", fontsize=20)


plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.show()

In [ ]:
%%time
fig, axes = plt.subplots(2, 2, figsize=(30, 20), dpi=120)
axes = axes.flatten()  # flatten 2D array of axes into a list

plot_settings = [
    ("co2flux_pre_reconstructed", "RdYlBu_r", -8, 8),
    ("co2flux_pre_simulated", "RdYlBu_r", -8, 8),
    ("reconstruction_error_signed", "PiYG", -3, 3),
    ("reconstruction_error_absolute", "Purples", 0, 3),
    
]

for ax, (plot_col, cmap, vmin, vmax) in zip(axes, plot_settings):
    world_map = Basemap(
        projection='cyl', resolution='c',
        llcrnrlat=-90, urcrnrlat=90,
        llcrnrlon=-180, urcrnrlon=180,
        ax=ax
    )

    sc = world_map.scatter(
        _df_['nav_lon'], _df_['nav_lat'],
        s=5, 
        c=_df_[plot_col],
        vmin=vmin, vmax=vmax,
        cmap=cmap, edgecolors='none'
    )

    world_map.fillcontinents(color='black')

    ax.set_title(f"{plot_col} in {required_year}-{required_month}", fontsize=20)

    cbar = fig.colorbar(sc, ax=ax, shrink=0.3, pad=0.02)
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label(f"{plot_col} in month {required_month}", fontsize=20)

# Reduce spacing between rows and columns
plt.tight_layout(h_pad=-25, w_pad=0.2)
plt.show()